# 🤝 Module 3: Collaborative Filtering & Matrix Factorization
**AI-Powered Personalized Product Recommendation & Analytics Platform**

In this notebook:
1. Weighted Implicit Feedback Matrix formulation
2. User-Item Interaction Matrix construction
3. Matrix Factorization with TruncatedSVD (Singular Value Decomposition)
4. Predicting user preference scores for unseen products
5. Market Basket Association Mining (Frequently Bought Together)


In [ ]:
import os
import sys
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

BASE_DIR = '..'
interactions_path = os.path.join(BASE_DIR, 'data', 'raw', 'interactions.csv')
products_path = os.path.join(BASE_DIR, 'data', 'processed', 'clean_products.csv')

df_interactions = pd.read_csv(interactions_path)
df_products = pd.read_csv(products_path)


### 1. Weighted Implicit Feedback


In [ ]:
interaction_weights = {
    'view': 1.0,
    'click': 2.0,
    'like': 3.0,
    'wishlist': 3.5,
    'cart': 4.0,
    'purchase': 5.0
}

df_weighted = df_interactions.copy()
df_weighted['weight'] = df_weighted['interaction_type'].map(lambda t: interaction_weights.get(t, 1.0))
df_weighted.head()


### 2. User-Item Matrix Formulation


In [ ]:
user_item_matrix = df_weighted.groupby(['user_id', 'product_id'])['weight'].sum().unstack().fillna(0.0)
print(f"User-Item Matrix shape: {user_item_matrix.shape}")
user_item_matrix.iloc[:5, :5]


### 3. Matrix Factorization via TruncatedSVD


In [ ]:
n_components = 8
svd = TruncatedSVD(n_components=n_components, random_state=42)
user_factors = svd.fit_transform(user_item_matrix.values)
item_factors = svd.components_

print(f"Explained Variance Ratio: {svd.explained_variance_ratio_.sum():.2%}")
print(f"User Factors shape: {user_factors.shape}")
print(f"Item Factors shape: {item_factors.shape}")

# Reconstruct predicted rating matrix
predicted_matrix = np.dot(user_factors, item_factors)
df_predictions = pd.DataFrame(predicted_matrix, index=user_item_matrix.index, columns=user_item_matrix.columns)
df_predictions.iloc[:5, :5]


### 4. Collaborative Recommendations for Target User


In [ ]:
target_user = 'USER_TECH_001'
user_preds = df_predictions.loc[target_user].sort_values(ascending=False).head(5)

rec_list = []
for pid, score in user_preds.items():
    p = df_products[df_products['product_id'] == pid].iloc[0]
    rec_list.append({
        'Product ID': pid,
        'Product Name': p['product_name'],
        'Category': p['category'],
        'Predicted Score': round(score, 3)
    })

pd.DataFrame(rec_list)


### 5. Co-occurrence Matrix (Frequently Bought Together)


In [ ]:
# High-intent events
cart_purchases = df_interactions[df_interactions['interaction_type'].isin(['cart', 'purchase', 'wishlist'])]
user_baskets = cart_purchases.groupby('user_id')['product_id'].unique()

from collections import defaultdict
co_matrix = defaultdict(lambda: defaultdict(int))
for pids in user_baskets:
    for i in range(len(pids)):
        for j in range(len(pids)):
            if i != j:
                co_matrix[pids[i]][pids[j]] += 1

df_co = pd.DataFrame(co_matrix).fillna(0)
print("Top items frequently bought with Apple iPhone 15 (PROD_ELEC_001):")
if 'PROD_ELEC_001' in df_co.columns:
    print(df_co['PROD_ELEC_001'].sort_values(ascending=False).head(3))
